# SHG at the surfaces of a crystal

Which components of the second-harmonic susceptibility χ_ijk are allowed in the bulk of a crystal,
and which at a surface? At a surface the symmetry is lowered to the operations of the point group
that leave the surface normal unchanged (its site symmetry), and the tensor form follows from those.

This notebook does it for a cubic crystal of class m-3m and the surfaces (001), (110), (111). The
numbers are checked against print in the tests of symcrys (Yariv, *Quantum Electronics*, 2nd ed.,
Table 16.1, for the classes 4mm, mm2 and 3m).

In [1]:
import numpy as np
import symcrys

symcrys.VERSION

'0.10.0'

The tensor: polar, rank 3, symmetric in its last two indices (χ_ijk = χ_ikj, the
index symmetry of SHG). Components are numbered by a flat index in C order, so a row of `basis`
reshaped to `(3, 3, 3)` is a tensor χ_ijk.

In [2]:
SHG = {"rank": 3, "parity": "polar", "intrinsic": "jk"}
AXES = "xyz"
name = lambda idx: "χ_" + "".join(AXES[i] for i in idx)

bulk = symcrys.tensor_form("m-3m", SHG)
print("bulk m-3m:", bulk["basis"].shape[0], "independent components")

bulk m-3m: 0 independent components


## Site symmetry of the surface normals

`site_symmetry` gives the operations of the point group (indices into `operations`) that leave a
direction fixed. The frame of each surface: z along the normal, x along the normal of a mirror
plane that contains it, y = z × x.

In [3]:
ops = symcrys.operations("m-3m")
surfaces = {
    "(001)": ([0, 0, 1], [1, 0, 0]),
    "(110)": ([1, 1, 0], [0, 0, 1]),
    "(111)": ([1, 1, 1], [1, -1, 0]),
}

def unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)

for surface, (normal, x) in surfaces.items():
    site = symcrys.site_symmetry(ops, normal)
    print(surface, "site symmetry:", len(site["operations"]), "operations, multiplicity", site["multiplicity"])

(001) site symmetry: 8 operations, multiplicity 6
(110) site symmetry: 4 operations, multiplicity 12
(111) site symmetry: 6 operations, multiplicity 8


## Tensor form at each surface, in the frame of the surface

The operations of the site symmetry, expressed in the frame of the surface (`frame` · W · `frame`ᵀ,
rows of `frame`: the new axes in the crystal frame), go into `tensor_form_of`, which gives the form
under any set of operations. Its free parameters are then components of the surface frame.

In [4]:
def surface_form(normal, x):
    site = symcrys.site_symmetry(ops, normal)
    z, xa = unit(normal), unit(x)
    frame = np.array([xa, np.cross(z, xa), z])
    in_frame = [{"matrix": frame @ ops[i]["matrix"] @ frame.T} for i in site["operations"]]
    return symcrys.tensor_form_of(in_frame, SHG), frame

for surface, (normal, x) in surfaces.items():
    form, frame = surface_form(normal, x)
    nonzero = sorted({tuple(np.unravel_index(i, (3, 3, 3))) for i in np.flatnonzero(np.any(form["basis"] != 0, axis=0))})
    nonzero = [idx for idx in nonzero if idx[1] <= idx[2]]  # χ_ijk = χ_ikj: list each pair once
    print(f"{surface}: {len(form['parameters'])} free parameters", ", ".join(name(i) for i in form["parameters"]))
    print("        nonzero components:", ", ".join(name(i) for i in nonzero))

(001): 3 free parameters χ_xxz, χ_zxx, χ_zzz
        nonzero components: χ_xxz, χ_yyz, χ_zxx, χ_zyy, χ_zzz
(110): 5 free parameters χ_xxz, χ_yyz, χ_zxx, χ_zyy, χ_zzz
        nonzero components: χ_xxz, χ_yyz, χ_zxx, χ_zyy, χ_zzz
(111): 4 free parameters χ_xxy, χ_xxz, χ_zxx, χ_zzz
        nonzero components: χ_xxy, χ_xxz, χ_yxx, χ_yyy, χ_yyz, χ_zxx, χ_zyy, χ_zzz


The (001) surface keeps 4mm (3 independent components), (110) keeps mm2 (5) and (111) keeps 3m
(4). For a simulation, `basis` gives every allowed tensor as Σₖ pₖ · basis[k], where pₖ is the value
of the parameter component `parameters[k]`; the reshaped rows go straight into `np.einsum`, e.g.
for the nonlinear polarization P_i = χ_ijk E_j E_k:

In [5]:
form, frame = surface_form([1, 1, 1], [1, -1, 0])
chi = form["basis"].reshape(-1, 3, 3, 3)
p = np.array([1.0, 0.5, 0.2, -0.3])          # values of the four parameters, arbitrary here
E = np.array([1.0, 0.0, 0.0])                # field along x of the surface frame
np.einsum("k,kijl,j,l->i", p, chi, E, E)

array([0. , 1. , 0.2])